# 01 - Map VASA onto Controls

In [ ]:
%autosave 0

In [ ]:
import os
import sys
from pathlib import Path

# Set the base project directory
base_proj_dir = Path("/projects/site/pred/ihb-g-deco/USERS/adaml9/intestine_fate/notebooks/tf_ko_panel_analysis")

# Append necessary paths for module imports
sys.path.append("/projects/site/pred/ihb-g-deco/USERS/adaml9/bioinfo-blueprint/src/python")
sys.path.append(str(base_proj_dir))

import numpy as np
import pandas as pd
import anndata as ad
import cellmapper
import scanpy as sc
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import marsilea as ma
import marsilea.plotter as mp
from dynaconf import Dynaconf

# Load settings
settings = Dynaconf(
    settings_files=[base_proj_dir / 'settings.toml', base_proj_dir / '.secrets.toml'],
)

sc.settings.figdir = settings.ANALYSIS.figures_dir

In [ ]:
data_dir = Path(settings.IO.combined_data) / "all-sample" / "DGE_filtered"
anndata_dir = Path(settings.IO.anndata)

In [ ]:
cellmapper.__version__

## Load data 

In [ ]:
adata = sc.read(anndata_dir / "tf_ko_panel_control_vasa_combined_pca_including_cystic_v3.h5ad")

In [ ]:
adata_query = adata[adata.obs["dataset"].isin(["TF_KO_Controls"])]
adata_ref = adata[adata.obs["dataset"].isin(["VASA"])]

In [ ]:
adata_query.shape, adata_ref.shape

In [ ]:
cmap = cellmapper.CellMapper(adata_query, adata_ref).map(
    obs_keys=["annotation", "milestones"], use_rep="X_pca", knn_method="pynndescent"
)
cmap

In [ ]:
sc.pl.embedding(adata_query, basis="X_umap", color=["annotation_pred", "annotation_conf", "milestones_pred", "milestones_conf"], legend_loc="on data")

In [ ]:
adata_parse = sc.read(anndata_dir / "tf_ko_panel_contrastiveVI_raw_annotated_eec.h5ad")
adata_parse_controls = adata_parse[adata_parse.obs["condition"].isin(["Control"])]

adata_parse_controls = adata_parse_controls.raw.to_adata()

# Normalize and log transform
sc.pp.normalize_total(adata_parse_controls, target_sum=1e4)
sc.pp.log1p(adata_parse_controls)
adata_parse_controls.layers["lognorm"] = adata_parse_controls.X.copy()

In [ ]:
adata_query.obs_names = adata_query.obs_names.str.replace("-TF_KO_Controls", "")

adata_parse_controls.obs["annotation_pred"] = adata_query.obs["annotation_pred"]
adata_parse_controls.obs["annotation_conf"] = adata_query.obs["annotation_conf"]
adata_parse_controls.obs["milestones_pred"] = adata_query.obs["milestones_pred"]
adata_parse_controls.obs["milestones_conf"] = adata_query.obs["milestones_conf"]

In [ ]:
adata_parse_controls.write(anndata_dir / "tf_ko_panel_control_mapped_vasa_milestones_including_cystic_v3.h5ad")